<a href="https://colab.research.google.com/github/SofiiaBobr/goit_machine_learning/blob/main/homework_3/lesson_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Тема 6. Робота з файлами та модульна система

## Мета уроку
Сьогодні ми розберемо три ключові аспекти роботи професійного розробника:
1.  **Читання та аналіз даних** з файлів (менеджери контексту).
2.  **Структурування даних** (перетворення рядків у словники).
3.  **Взаємодія з користувачем** (створення CLI-бота).

Ми будемо дотримуватися принципу: *логіка відокремлена від інтерфейсу*.



## Завдання 1: Аналіз зарплат співробітників

**Теоретична довідка:**
Для безпечного читання файлів використовуємо конструкцію `with open(...)`. Вона гарантує закриття файлу навіть при помилках.
Дані у файлі часто розділені комами (CSV-подібний формат). Метод `split(',')` допоможе розділити рядок на частини.

**Умова задачі:**
Створіть функцію `total_salary(path)`, яка:
1.  Приймає шлях до файлу.
2.  Читає дані (прізвище, зарплата).
3.  Повертає кортеж: `(загальна сума, середня зарплата)`.
4.  Обробляє винятки (якщо файлу немає або він пошкоджений).



In [ ]:
def total_salary(path):
    try:
        total = 0
        count = 0
        with open(path, 'r', encoding='utf-8') as file:
          for line in file:
            line = line.strip()
            if not line:
              continue
            try:
                name, salary_str = line.split(',')
                total += float(salary_str)
                count +=1
            except ValueError:
                print(f"Помилка формату в рядку: '{line}'. Пропускаємо.")
                continue

        if count == 0:
          return 0, 0
        average = total/count

        return total, average
    except FileNotFoundError:
        print("Файл не знайдено.")
        return 0, 0
    except Exception as e:
        print(f"Неочікувана помилка: {e}")
        return 0, 0

In [ ]:
total, average = total_salary("data.txt")
print(f' загальна сума заробітньої плати: {total}. середня заробітня плата: {average}. ')


 загальна сума заробітньої плати: 160000.0. середня заробітня плата: 40000.0. 


## Завдання 2: Робота зі структурованими даними (Коти)

**Теоретична довідка:**
Часто нам потрібно перетворити "сирі" дані з файлу у зручний для програми формат — наприклад, список словників.
Кожен рядок файлу: `id, name, age`.
Результат має бути списком: `[{"id": "...", "name": "...", "age": "..."}, ...]`.

**Умова задачі:**
Реалізуйте функцію `get_cats_info(path)`, яка повертає список словників.



In [ ]:
def get_cats_info(path):
    cats_list = []
    try:
      with open(path, 'r', encoding='utf-8') as file:
        for line in file:
            line = line.strip()
            if not line:
              continue
            try:
                id, name, age = line.split(',')
                cat_data = {
                    "id": id,
                    "name": name,
                    "age": age
                }
                cats_list.append(cat_data)
            except ValueError:
                print(f"Помилка формату в рядку: '{line}'. Пропускаємо.")
                continue
    except FileNotFoundError:
        print("Файл не знайдено.")
    except Exception as e:
        print(f"Неочікувана помилка: {e}")
    return cats_list




In [ ]:

get_cats_info("listcats.txt")

Файл не знайдено.


[]

## Завдання 3: Візуалізація структури директорії

**Теоретична довідка:**
Для роботи зі шляхами використовуємо модуль `pathlib`.
`Path.iterdir()` дозволяє пройтись по вмісту папки.
`Path.is_dir()` та `Path.is_file()` перевіряють тип об'єкта.

**Умова:**
Напишіть скрипт, який приймає шлях до директорії та виводить її вміст. Бажано використати кольори (бібліотека `colorama`) для розрізнення папок та файлів.



In [ ]:
import sys
from pathlib import Path
from colorama import Fore, Style, init

init(autoreset=True)

def scan(path, indent=""):
    p = Path(path)
    for item in p.iterdir():
        if item.is_dir():
            print(f"{indent}{Fore.BLUE}{item.name}/")
            scan(item, indent+"    ")
        else:
             print(f"{indent}{Fore.GREEN}{item.name}")


if __name__=="__main__":
    scan(sys.argv[1])


## Завдання 4: Консольний бот-асистент

**Архітектура:**
1.  **Парсер (`parse_input`):** Розбиває введений рядок на команду та аргументи. Приводить команду до нижнього регістру (`strip()`, `lower()`).
2.  **Функції-хендлери (`add_contact`, `change_contact`, `show_phone`, `show_all`):** Виконують конкретну дію.
3.  **Цикл (`main`):** Безкінечний цикл `while True`, який очікує вводу.

**Структура команд:**
* `add [name] [phone]` - додати контакт.
* `change [name] [phone]` - змінити номер.
* `phone [name]` - показати номер.
* `all` - показати всі.
* `hello` - привітання.
* `exit` / `close` - вихід.



In [ ]:
from logging import Handler
def parse_input(user_input):
    cmd, *args = user_input.split()
    cmd = cmd.strip().lower()
    return cmd, *args

def add_contact(args, contacts):
    if len(args) != 2:return "Error: Give me name and phone please."
    name, phone = args
    contacts[name] = phone
    return "Contact added."

def change_contact(args, contacts):
    if len(args) != 2:return "Error: Give me name and phone please."
    name, phone = args
    if name in contacts:
      contacts[name] = phone
      return "Contact updated."

def show_phone(args, contacts):
    if len(args) != 1:return "Error: Give me name."
    name = args[0]
    if name in contacts:return contacts[name]


def show_all(args, contacts):
    if not contacts:return "No contact saved"
    return "\n".join([f"{name}: {phone}" for name, phone in contacts.items()])

def hello(args, contacts):
  return "How can I help you?"

COMMAND = {
    "add": add_contact,
    "hello": hello,
    "change": change_contact,
    "phone": show_phone,
    "all": show_all
}

def main():
    contacts = {}
    print("Welcome to the assistant bot!")

    while True:
        user_input = input("Enter a command: ").strip()
        if not user_input:
            continue

        command, *args = parse_input(user_input)


        if command in ["close", "exit"]:
            print("Good bye!")
            break
        if command in COMMAND:
          HandlerFunction = COMMAND[command]
          print(HandlerFunction(args,contacts))
        else:
          print("Invalide command")

if __name__ == "__main__":
     main()


Welcome to the assistant bot!
Enter a command: add sofi 64793
Contact added.
Enter a command: add denys 34685
Contact added.
Enter a command: all
sofi: 64793
denys: 34685
Enter a command: phone sofi
64793
Enter a command: change denys 999999999
Contact updated.
Enter a command: all
sofi: 64793
denys: 999999999
Enter a command: hello
How can I help you?
Enter a command: exit
Good bye!


In [ ]:
def caching_fibonacci():
    # 1. Створи порожній словник cache
    cache = {}

    def fibonacci(n):
        # 2. Базові випадки: якщо n <= 0 поверни 0, якщо n == 1 поверни 1

        # 3. Перевірка кешу: якщо n вже є в cache, поверни cache[n]

        # 4. Рекурсія: cache[n] = fibonacci(n-1) + ...

        # 5. Поверни cache[n]
        if n <=0: return 0
        if n ==1: return 1
        if n in cache:
            return cache[n]
        cache[n]=fibonacci(n-1)+fibonacci(n-2)
        return cache[n]

    return fibonacci

In [ ]:
fib = caching_fibonacci()
print(fib(10))  # Має вивести 55
print(fib(15))  # Має вивести 610

55
610


In [ ]:
import re
from typing import Callable, Generator

def generator_numbers(text: str) -> Generator[float, None, None]:

    # 1. Створи патерн для пошуку чисел (врахуй пробіли навколо, якщо треба)
    pattern = r" \d+\.\d+ "

    # 2. Використай re.finditer або re.findall
    for word in text.split():
        matches = re.findall(r"\d+\.\d+", text)
        for match in matches:
            yield float(match)
        break
    # 3. Пройдись по знайдених елементах і зроби yield (перетворивши на float)


def sum_profit(text: str, func: Callable):
    # 1. Виклич func(text), щоб отримати генератор
    sum = 0
    for num in func(text):
        sum += num
    return sum
    # 2. Просумуй елементи (можна функцією sum() або циклом)

In [ ]:
text = "Загальний дохід працівника складається з декількох частин: 1000.01 як основний дохід, доповнений додатковими надходженнями 27.45 і 324.00 доларів."
total_income = sum_profit(text, generator_numbers)
print(f"Загальний дохід: {total_income}")

Загальний дохід: 1351.46
